In [ ]:
# Import python packages
import streamlit as st
import pandas as pd

# We can also use Snowpark for our analyses!
from snowflake.snowpark.context import get_active_session
session = get_active_session()


In [ ]:
-- VQA Quickstart Setup
-- Run this script to set up the required Snowflake objects

-- 1. Create Database and Schema
CREATE DATABASE IF NOT EXISTS VQA_DEMO_DB;
CREATE SCHEMA IF NOT EXISTS VQA_DEMO_DB.VQA;

USE DATABASE VQA_DEMO_DB;
USE SCHEMA VQA;

-- 2. Create External Stage for S3 source data
CREATE OR REPLACE STAGE VQA_DEMO_DB.VQA.S3_DATA_STAGE
    URL = 's3://sfquickstarts/sfguide_ml_batch_inference_with_vision_models/'
    DIRECTORY = (ENABLE = TRUE AUTO_REFRESH = FALSE);

-- 3. Create Internal Stages for data and images
CREATE STAGE IF NOT EXISTS VQA_DEMO_DB.VQA.DATA_STAGE
    DIRECTORY = (ENABLE = TRUE)
    ENCRYPTION = (TYPE = 'SNOWFLAKE_SSE');

CREATE STAGE IF NOT EXISTS VQA_DEMO_DB.VQA.IMAGES_STAGE
    DIRECTORY = (ENABLE = TRUE)
    ENCRYPTION = (TYPE = 'SNOWFLAKE_SSE');

-- 4. Create Compute Pool for GPU inference
CREATE COMPUTE POOL IF NOT EXISTS GPU_INFERENCE_POOL
    MIN_NODES = 1
    MAX_NODES = 1
    INSTANCE_FAMILY = GPU_NV_M
    AUTO_RESUME = TRUE
    AUTO_SUSPEND_SECS = 300;

-- 5. Create CSV file format for S3 data
CREATE FILE FORMAT IF NOT EXISTS VQA_DEMO_DB.VQA.CSV_FORMAT
    TYPE = 'CSV'
    SKIP_HEADER = 1
    FIELD_OPTIONALLY_ENCLOSED_BY = '"'
    ESCAPE_UNENCLOSED_FIELD = NONE;

-- 6. Create table for inference results  
CREATE TABLE IF NOT EXISTS VQA_DEMO_DB.VQA.INFERENCE_RESULTS (
    ID INT,
    QUESTION VARCHAR,
    EXPECTED_ANSWER VARCHAR,
    PREDICTED_ANSWER VARCHAR,
    QUESTION_TYPE VARCHAR,
    IS_CORRECT BOOLEAN
);

-- 7. Create file format for parquet output
CREATE FILE FORMAT IF NOT EXISTS VQA_DEMO_DB.VQA.PARQUET_FORMAT
    TYPE = 'PARQUET';

-- Verify setup
SHOW STAGES IN SCHEMA VQA_DEMO_DB.VQA;
SHOW COMPUTE POOLS LIKE 'GPU_INFERENCE_POOL';

SELECT 'Setup complete! You can now proceed with the notebooks.' AS STATUS;

## Define S3 Source Paths

In [ ]:
S3_BUCKET_URL = 's3://sfquickstarts/sfguide_ml_batch_inference_with_vision_models/'

print(f"S3 source: {S3_BUCKET_URL}")

## Create External Stage for S3 Data

In [ ]:
session.sql(f"CREATE OR REPLACE STAGE S3_DATA_STAGE URL='{S3_BUCKET_URL}' DIRECTORY = (ENABLE = TRUE AUTO_REFRESH = FALSE)").collect()

print("Created external stage S3_DATA_STAGE")
session.sql("LS @S3_DATA_STAGE/IMAGES/").show(5)

## Copy Images to Internal Stage

In [ ]:
session.sql("CREATE OR REPLACE STAGE IMAGES_STAGE DIRECTORY = (ENABLE = TRUE) ENCRYPTION = (TYPE = 'SNOWFLAKE_SSE')").collect()

session.sql("COPY FILES INTO @IMAGES_STAGE/ FROM @S3_DATA_STAGE/IMAGES/").collect()

print("Copied images from S3 to internal stage")
result = session.sql("LS @IMAGES_STAGE").collect()
print(f"Total images in stage: {len(result)}")

## Load VQA Dataset

In [ ]:
import json
import pandas as pd

session.sql("""
CREATE OR REPLACE FILE FORMAT CSV_FORMAT
    TYPE = 'CSV'
    SKIP_HEADER = 1
    FIELD_OPTIONALLY_ENCLOSED_BY = '"'
    ESCAPE_UNENCLOSED_FIELD = NONE
""").collect()

df_vqa = session.sql("""
SELECT 
    $1::INT AS ID,
    $2::INT AS IMAGE_IDX,
    $3::STRING AS IMAGE_PATH,
    $4::STRING AS QUESTION,
    $5::STRING AS ANSWER,
    $6::STRING AS QUESTION_TYPE
FROM @S3_DATA_STAGE/vqa_dataset.csv (FILE_FORMAT => 'CSV_FORMAT')
""").to_pandas()

num_images = df_vqa['IMAGE_IDX'].nunique()
print(f"Created {len(df_vqa)} Q&A records from {num_images} images")
print(f"\nSample questions:")
for i in range(min(5, len(df_vqa))):
    print(f"  Q: {df_vqa.iloc[i]['QUESTION']}")
    print(f"  A: {df_vqa.iloc[i]['ANSWER']}\n")

In [ ]:
df_sf = session.create_dataframe(df_vqa)
df_sf.write.mode("overwrite").save_as_table(f"VQA_DEMO_DB.VQA.VQA_DATASET")

print(f"Saved dataset to VQA_DEMO_DB.VQA.VQA_DATASET")
session.table(f"VQA_DEMO_DB.VQA.VQA_DATASET").show(5)

## Format Input for Batch Inference

In [ ]:
def create_chat_message(row):
    return json.dumps([
        {
            "role": "user",
            "content": [
                {"type": "text", "text": f"{row['QUESTION']} Answer briefly in a few words."},
                {"type": "image_url", "image_url": {"url": row['IMAGE_PATH']}}
            ]
        }
    ])

df_vqa['MESSAGES'] = df_vqa.apply(create_chat_message, axis=1)

input_df = session.create_dataframe(df_vqa[['ID', 'MESSAGES', 'QUESTION', 'ANSWER', 'QUESTION_TYPE']])
input_df.write.mode("overwrite").save_as_table(f"VQA_DEMO_DB.VQA.INFERENCE_INPUT")

print(f"Prepared {input_df.count()} samples for inference")
print("\nSample message format:")
print(json.dumps(json.loads(df_vqa.iloc[0]['MESSAGES']), indent=2))

# Batch Inference with VLM on OCR-VQA

In this section, you'll run batch inference using your imported HuggingFace model on book cover images and evaluates text reading accuracy.

In [ ]:
from snowflake.ml.registry import Registry

MODEL_NAME = "LLAVA_V1_6_MISTRAL_7B_HF"  # Updated to match actual model name
MODEL_VERSION = "v1"

reg = Registry(session=session, database_name='VQA_DEMO_DB', schema_name='VQA')

In [ ]:
model = reg.get_model(MODEL_NAME)
mv = model.default

print(f"Model: {MODEL_NAME}")
print(f"Version: {mv._version_name}")
print(f"Functions: {[f['name'] for f in mv.show_functions()]}")

In [ ]:
input_table = session.table(f"VQA_DEMO_DB.VQA.INFERENCE_INPUT")
test_data = input_table.to_pandas()

print(f"Loaded {len(test_data)} samples for inference")
print(f"\nQuestion types:")
print(test_data['QUESTION_TYPE'].value_counts())

## Run Batch Inference

In [ ]:
from snowflake.ml.model.batch import JobSpec, OutputSpec, SaveMode, InputSpec
import time

OUTPUT_STAGE = f"@VQA_DEMO_DB.VQA.DATA_STAGE/inference_output/"
COMPUTE_POOL = "GPU_INFERENCE_POOL"

test_data_subset = test_data.head(10)
input_df = session.create_dataframe(test_data_subset[['MESSAGES']])

print(f"Starting batch inference on {input_df.count()} samples...")
print(f"Output: {OUTPUT_STAGE}")
start_time = time.time()

job = mv.run_batch(
    compute_pool=COMPUTE_POOL,
    X=input_df,
    input_spec=InputSpec(params={"temperature": 0.1, "max_completion_tokens": 100}),
    output_spec=OutputSpec(stage_location=OUTPUT_STAGE, mode=SaveMode.OVERWRITE),
    job_spec=JobSpec(gpu_requests="1")
)

print("Job submitted. Waiting for completion...")
job.wait()

elapsed = time.time() - start_time
print(f"\nCompleted in {elapsed:.1f}s ({elapsed/len(test_data_subset):.1f}s per sample)")

## Load and Parse Results

In [ ]:
results_df = session.read.option("pattern", ".*\\.parquet").parquet(OUTPUT_STAGE)
results_pd = results_df.to_pandas()

print(f"Loaded {len(results_pd)} results")
print(f"Columns: {results_pd.columns.tolist()}")

def extract_prediction(row):
    try:
        choices = row.get('id')
        if isinstance(choices, str):
            choices = json.loads(choices)
        if choices and len(choices) > 0:
            return choices[0]['message']['content'].strip()
    except:
        pass
    return None

results_pd['PREDICTION'] = results_pd.apply(extract_prediction, axis=1)

print("\nSample predictions:")
for i in range(min(3, len(results_pd))):
    print(f"  {i+1}: {results_pd.iloc[i]['PREDICTION'][:80]}...")

## Evaluate Results

In [ ]:
import re

def normalize(text):
    if not text:
        return ""
    text = str(text).lower().strip()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    return text

def is_correct(predicted, expected, q_type):
    if not predicted:
        return False
    
    pred = normalize(predicted)
    exp = normalize(expected)
    
    if exp in pred:
        return True
    if pred.startswith(exp):
        return True
    for word in exp.split():
        if len(word) > 3 and word in pred:
            return True
    return False

eval_data = test_data_subset.copy()
eval_data['PREDICTED'] = results_pd['PREDICTION'].values
eval_data['IS_CORRECT'] = eval_data.apply(
    lambda r: is_correct(r['PREDICTED'], r['ANSWER'], r['QUESTION_TYPE']), axis=1
)

accuracy = eval_data['IS_CORRECT'].mean() * 100

print("=" * 50)
print("OCR-VQA EVALUATION RESULTS")
print("=" * 50)
print(f"Overall Accuracy: {accuracy:.1f}%")
print(f"Correct: {eval_data['IS_CORRECT'].sum()}/{len(eval_data)}")
print("=" * 50)

## View Detailed Results

In [ ]:
print("\nDetailed Results:")
print("-" * 80)

for _, row in eval_data.iterrows():
    status = "✓" if row['IS_CORRECT'] else "✗"
    print(f"\n{status} Q: {row['QUESTION']}")
    print(f"   Expected: {row['ANSWER']}")
    print(f"   Got: {str(row['PREDICTED'])[:100]}")

## Save Results

In [ ]:
results_to_save = eval_data[['ID', 'QUESTION', 'ANSWER', 'PREDICTED', 'QUESTION_TYPE', 'IS_CORRECT']].copy()
results_to_save.columns = ['ID', 'QUESTION', 'EXPECTED_ANSWER', 'PREDICTED_ANSWER', 'QUESTION_TYPE', 'IS_CORRECT']

session.create_dataframe(results_to_save).write.mode("overwrite").save_as_table(
    f"VQA_DEMO_DB.VQA.INFERENCE_RESULTS"
)

print(f"Results saved to VQA_DEMO_DB.VQA.INFERENCE_RESULTS")